In [1]:
import pandas as pd
import numpy as np
import warnings
import gc
import os
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
from itertools import combinations
from xgboost import XGBClassifier
from scipy.special import expit
from tqdm import tqdm

pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option("display.max_colwidth", None)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
TARGET = 'loan_paid_back'
NUMS = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
CATS = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']

In [6]:
train = pd.read_csv('train.csv', index_col='id')
test = pd.read_csv('test.csv', index_col='id')
bin_features_train = pd.DataFrame(index=train.index)
bin_features_test = pd.DataFrame(index=test.index)

for c in NUMS:
    for q in [5]:
        try:
            train_bins, bins = pd.qcut(train[c], q=q, labels=False, retbins=True, duplicates="drop")
            bin_features_train[f"{c}_bin{q}"] = train_bins
            bin_features_test[f"{c}_bin{q}"] = pd.cut(test[c], bins=bins, labels=False, include_lowest=True)
        except Exception:
            bin_features_train[f"{c}_bin{q}"] = 0
            bin_features_test[f"{c}_bin{q}"] = 0
train = pd.concat([train, bin_features_train], axis=1)
test = pd.concat([test, bin_features_test], axis=1)


In [9]:
train['default_risk'] = (train['debt_to_income_ratio'] * 0.40 + (850 - train['credit_score']) / 850 * 0.35 + train['interest_rate'] / 100 * 0.25)
test['default_risk'] = (test['debt_to_income_ratio'] * 0.40 + (850 - test['credit_score']) / 850 * 0.35 + test['interest_rate'] / 100 * 0.25)
for c in ['credit_score']:
    n = f'{c}2'
    train[n] = train[c].copy()
    test[n] = test[c].copy()




In [12]:
DIGITS = []
for c in ['annual_income', 'loan_amount']:
    for k in range(-4, 2):
        n = f'{c}_d{k}'
        train[n] = ((train[c] * 10**k) % 10).fillna(-1).astype("int8")
        test[n] = ((test[c] * 10**k) % 10).fillna(-1).astype("int8")
        DIGITS.append(n)

for c in ['interest_rate']:
    for k in range(-1, 3):
        n = f'{c}_d{k}'
        train[n] = ((train[c] * 10**k) % 10).fillna(-1).astype("int8")
        test[n] = ((test[c] * 10**k) % 10).fillna(-1).astype("int8")
        DIGITS.append(n)

for c in ['debt_to_income_ratio']:
    for k in range(1, 4):
        n = f'{c}_d{k}'
        train[n] = ((train[c] * 10**k) % 10).fillna(-1).astype("int8")
        test[n] = ((test[c] * 10**k) % 10).fillna(-1).astype("int8")
        DIGITS.append(n)

train['grade_subgrade_d1'] = train['grade_subgrade'].apply(lambda x: x[1]).astype('int8')
test['grade_subgrade_d1'] = test['grade_subgrade'].apply(lambda x: x[1]).astype('int8')

In [19]:
ROUND = []
RR = [-1, 0]
for c in ['annual_income', 'loan_amount']:
    for r in RR:
        n = f"{c}_r{r}"
        train[n] = train[c].round(r)
        test[n] = test[c].round(r)
        ROUND.append(n)

for c in CATS + ['credit_score2']:
    combined = pd.concat([train[c], test[c]])
    combined, _ = combined.factorize()
    train[c] = combined[:len(train)]
    test[c] = combined[len(train):len(train) + len(test)]


In [ ]:
TE_columns = []
CE_columns = []
PAIRS = []

columns = NUMS + CATS + [ROUND[0]]

columns = NUMS + CATS + [ROUND[0]]
# BIGRAMS
for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            # Smash two columns together into a string
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            # Smash two columns together into a string
            test[name] = test[name] + '_' + test[col].astype(str)


        combined = pd.concat([train[name], test[name]], ignore_index=True)
        # Label Encoding
        combined, _ = combined.factorize()
        if pd.Series(combined).nunique() > len(combined) // 2:
            train = train.drop(name, axis=1)
            test = test.drop(name, axis=1)
            continue
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        TE_columns.append(name)
        CE_columns.append(name)
        PAIRS.append(name)


100%|██████████| 66/66 [00:35<00:00,  1.86it/s]


In [41]:
for c1 in DIGITS[:6]:
    for c2 in ['employment_status', 'debt_to_income_ratio']:
        name = f'{c1}-{c2}'
        train[name] = train[c1].astype(str) + '_' + train[c2].astype(str)
        test[name] = test[c1].astype(str) + '_' + test[c2].astype(str)
    
        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]

        TE_columns.append(name)

for c1 in DIGITS[:6]:
    for c2 in [DIGITS[6], DIGITS[7]]:
        name = f'{c1}-{c2}'
        train[name] = train[c1].astype(str) + '_' + train[c2].astype(str)
        test[name] = test[c1].astype(str) + '_' + test[c2].astype(str)

        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]

        TE_columns.append(name)

In [42]:
for cols in tqdm([['annual_income', 'gender', 'marital_status']]):
    name = '-'.join(cols)

    train[name] = train[cols[0]].astype(str)
    for col in cols[1:]:
        train[name] = train[name] + '_' + train[col].astype(str)

    test[name] = test[cols[0]].astype(str)
    for col in cols[1:]:
        test[name] = test[name] + '_' + test[col].astype(str)

    combined = pd.concat([train[name], test[name]], ignore_index=True)
    combined, _ = combined.factorize()
    train[name] = combined[:len(train)]
    test[name] = combined[len(train):len(train) + len(test)]

    TE_columns.append(name)

100%|██████████| 1/1 [00:00<00:00,  1.18it/s]


In [49]:
TE_ORIG = []
CC = CATS + NUMS + DIGITS[:16]

from sklearn.model_selection import StratifiedKFold

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CC = CATS + NUMS + DIGITS[:16]

print(f"Processing {len(CC)} columns using K-Fold TE... ", end="")

for i, c in enumerate(CC):
    if i % 10 == 0:
        print(f"{i}, ", end="")
    
    # -----------------------------
    # A) Create empty TE column
    # -----------------------------
    train[f'TE_KFOLD_{c}'] = np.nan
    test[f'TE_KFOLD_{c}'] = 0
    
    global_mean = train[TARGET].mean()
    
    # -----------------------------
    # B) OUT-OF-FOLD TARGET ENCODING
    # -----------------------------
    for train_idx, val_idx in kf.split(train, train[TARGET]):
        X_tr, X_val = train.iloc[train_idx], train.iloc[val_idx]
        
        # Compute fold means
        means = X_tr.groupby(c)[TARGET].mean()
        
        # Map to validation fold
        train.loc[val_idx, f'TE_KFOLD_{c}'] = train.loc[val_idx, c].map(means)
    
    # Replace missing values (unseen categories)
    train[f'TE_KFOLD_{c}'] = train[f'TE_KFOLD_{c}'].fillna(global_mean)
    
    # -----------------------------
    # C) TARGET ENCODE TEST USING FULL TRAIN
    # -----------------------------
    full_means = train.groupby(c)[TARGET].mean()
    test[f'TE_KFOLD_{c}'] = test[c].map(full_means).fillna(global_mean)
    
    # -----------------------------
    # D) COUNT ENCODING (CE replacement)
    # -----------------------------
    counts = train[c].value_counts()
    train[f'CE_{c}'] = train[c].map(counts).fillna(0)
    test[f'CE_{c}'] = test[c].map(counts).fillna(0)

CC = CATS + NUMS

print(f"Processing {len(CC)} columns... ",end="")
print(f"Processing employment_status TE for {len(CATS + NUMS)} columns... ", end="")

for i, c in enumerate(CATS + NUMS):
    if i % 10 == 0:
        print(f"{i}, ", end="")
    
    global_mean = train['employment_status'].mean()
    
    train[f'TE_emp_{c}'] = np.nan
    test[f'TE_emp_{c}'] = 0
    
    # -----------------------------
    # A) Out-of-Fold TE (employment target)
    # -----------------------------
    for tr_idx, val_idx in kf.split(train, train[TARGET]):  
        # folds remain stratified on loan_paid_back
        X_tr, X_val = train.iloc[tr_idx], train.iloc[val_idx]
        
        means = X_tr.groupby(c)['employment_status'].mean()
        
        train.loc[val_idx, f'TE_emp_{c}'] = train.loc[val_idx, c].map(means)
    
    # Missing fill
    train[f'TE_emp_{c}'] = train[f'TE_emp_{c}'].fillna(global_mean)
    
    # -----------------------------
    # B) Apply TE to TEST using full train
    # -----------------------------
    full_means = train.groupby(c)['employment_status'].mean()
    test[f'TE_emp_{c}'] = test[c].map(full_means).fillna(global_mean)
print()

DIGIT_PAIRS = []
for r in [2, 3, 4]:
    for cols in tqdm(list(combinations(DIGITS[:6], r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        DIGIT_PAIRS.append(name)
        TE_columns.append(name)
        CE_columns.append(name)

for r in [2, 3, 4]:
    for cols in tqdm(list(combinations(DIGITS[6:12], r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)


        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        DIGIT_PAIRS.append(name)
        TE_columns.append(name)
        CE_columns.append(name)

for r in [2, 3, 4]:
    for cols in tqdm(list(combinations(DIGITS[12:16], r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)




        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        DIGIT_PAIRS.append(name)
        TE_columns.append(name)
        CE_columns.append(name)

for r in [2, 3]:
    for cols in tqdm(list(combinations(DIGITS[16:19], r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        combined = pd.concat([train[name], test[name]], ignore_index=True)
        combined, _ = combined.factorize()
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        DIGIT_PAIRS.append(name)
        TE_columns.append(name)
        CE_columns.append(name)

for c in NUMS + CATS:
    if c != 'employment_status':
        tmp = train.groupby(c)['employment_status'].mean()
        tmp.name = f'TE_mean_(employment_status)_{c}'
        train = train.merge(tmp, on=c, how='left')
        train[tmp.name] = train[tmp.name].fillna(train[tmp.name].mean())
        test = test.merge(tmp, on=c, how='left')
        test[tmp.name] = test[tmp.name].fillna(train[tmp.name].mean())

for c in NUMS + CATS:
    if c != 'debt_to_income_ratio':
        tmp = train.groupby(c)['debt_to_income_ratio'].mean()
        tmp.name = f'TE_mean_(debt_to_income_ratio)_{c}'
        train = train.merge(tmp, on=c, how='left')
        train[tmp.name] = train[tmp.name].fillna(train[tmp.name].mean())
        test = test.merge(tmp, on=c, how='left')
        test[tmp.name] = test[tmp.name].fillna(train[tmp.name].mean())

for c in test.columns.tolist():
    if test[c].dtype == 'float64':
        train[c] = train[c].astype('float32')
        test[c] = test[c].astype('float32')
    if test[c].dtype == 'int64':
        train[c] = train[c].astype('int32')
        test[c] = test[c].astype('int32')

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

Processing 27 columns using K-Fold TE... 0, 10, 20, Processing 11 columns... Processing employment_status TE for 11 columns... 0, 10, 


100%|██████████| 1/1 [00:00<00:00,  1.93it/s]
